In [ ]:
import time
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import numpy as np

In [ ]:
# Thực phẩm chức năng
def get_product(soup):
    try:
        product = soup.find("h1", attrs={"data-test":'product_name'})

        product_value = product.text
        
        product_string = product_value.strip()
        
    except AttributeError:
        product_string = "NULL"
    
    return product_string


def get_category(soup):
    try:
        category = soup.find("a", attrs ={"class": "text-blue-500"})

        category_value = category.text
        
        category_string = category_value.strip()
        
    except AttributeError:
        category_string = "NULL"
    
    return category_string


def get_price(soup):
    try:
        price = soup.find("span", attrs={"data-test": 'price'})

        price_value = price.text
        a = price_value.removesuffix("đ")
        
        price_string = a.strip()
        
        
        
    except AttributeError:
        price_string = "NULL"
    
    return price_string


def get_brand(soup):
    try:
        elements = soup.find_all("div", attrs ={"class":"css-1e2qim1 text-gray-10"})
        
        if len(elements) > 4:
            Dosage = elements[0].text  # Dosage
            brand_origin = elements[2].text   # Xuất xứ thương hiệu
            country = elements[3].text #xuẩt xứ thương hiệu 
        else:
            Dosage = brand_origin = country=  "NULL"
        
        return brand_origin
        
    except AttributeError:
        elements = "NULL"
    
    return elements


def get_trademark(soup):
    try:
        
        trademark = soup.find("a", attrs ={"class": "text-blue-5"})  
        if not trademark:
            trademark = soup.find("span", attrs={"class": "text-body2 md:text-label1"})
        
        
            if trademark:
            # Get the text and split it by space, then take the second element
                trademark_text = trademark.text.split(" ")[1]
            else:
                trademark_text = "NULL"
        trademark_value = trademark.text
        
        trademark_string = trademark_value.strip()
        
    except AttributeError:
        trademark_string = "NULL"
    
    return trademark_string


def get_country(soup):
    try:
        elements = soup.find_all("div", attrs ={"class":"css-1e2qim1 text-gray-10"})
        
        if len(elements) > 5:
            Dosage = elements[0].text  # Dosage
            brand_origin = elements[1].text   # Xuất xứ thương hiệu
            country = elements[4].text #xuẩt xứ thương hiệu 
        else:
            Dosage = brand_origin = country=  "NULL"
        
        return country
        
    except AttributeError:
        elements = "NULL"
    
    return elements


def get_rating(soup):
    try:
        rating = soup.find("span", attrs ={"class":"text-body2 text-gray-7 inline-flex items-center"})
        
        rating_value = rating.text
        
        rating_string = rating_value.strip()
        
    except AttributeError:
        rating_string ="NULL"
    
    return rating_string


def get_dosage(soup):
    try:
        elements = soup.find_all("div", attrs ={"class":"css-1e2qim1 text-gray-10"})
        
        if len(elements) > 3:
            Dosage = elements[0].text  # Dosage
            brand_origin = elements[1].text   # Xuất xứ thương hiệu
            country = elements[3].text #xuẩt xứ thương hiệu 
        else:
            Dosage = brand_origin = country=  "NULL"
        
        return Dosage
        
    except AttributeError:
        elements = "NULL"
    
    return elements




In [ ]:

if __name__ == '__main__':
    
    HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept': '*/*',
    'Connection': 'keep-alive'
}
    
    # Set up the Chrome WebDriver
    chrome_options = Options()  
    chrome_options.add_argument("--headless")
    
  
   

    driver = webdriver.Chrome(options=chrome_options)
    
    #URL of the mainpage
    
    URL = "https://nhathuoclongchau.com.vn/thuc-pham-chuc-nang"
    driver.get(URL)
    
    
    # Load more products until no more button is found
    while True:
        try:
            # Wait for the load more button to be clickable
            load_more_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'mt-3') and contains(@class, 'flex') and contains(@class, 'w-full') and contains(@class, 'items-center') and contains(@class, 'justify-center')]"))
            )
            load_more_button.click()
            time.sleep(2)  # Allow time for products to load
        except:
            break
    
    # Lấy mã nguồn HTML sau khi đã tải hết sản phẩm
    page_source = driver.page_source
    driver.quit()  

    # Create a soup object
    soup = BeautifulSoup(page_source, "html.parser")

    # Fetch all product links
    links = soup.find_all("a", attrs={'class': 'block px-3'})
    links_list = [link.get('href') for link in links if link.get('href')]

    # Dictionary to store product information
    data = {"Product name": [], "Category": [], "Price": [], "Brand origin": [], "Trademark": [], "Country": [], "Rating": [], "Dosage form": []}

    # Iterate through each link and fetch product details


    # Iterate through each link and fetch product details
    for link in links_list:
        product_url = "https://nhathuoclongchau.com.vn" + link
        new_webpage = requests.get(product_url, headers=HEADERS)
        new_soup = BeautifulSoup(new_webpage.content, "html.parser")

        # Append each piece of data to the dictionary
        data['Product name'].append(get_product(new_soup))
        data['Category'].append(get_category(new_soup))
        data['Price'].append(get_price(new_soup))
        data['Brand origin'].append(get_brand(new_soup))
        data['Trademark'].append(get_trademark(new_soup))
        data['Country'].append(get_country(new_soup))
        data['Rating'].append(get_rating(new_soup))
        data['Dosage form'].append(get_dosage(new_soup))

    LongChau = pd.DataFrame.from_dict(data)
    LongChau['Product name'].replace('', np.nan, inplace = True)
    LongChau = LongChau.dropna(subset=['Product name'])
    LongChau.to_csv("LongChau.csv", header=True, index = False)